# Inference Test Data (Resumable)

This notebook follows `inference.py` step by step and adds:
- batched inference from `csv/lrs2_test_modified.csv`
- resume/checkpointing for interrupted runs
- `tqdm` progress tracking
- optional Weights & Biases (`wandb`) logging for per-row and mean WER

## tmux quick usage

```bash
tmux new -s usr2_infer
jupyter lab --no-browser --port 8899
# detach: Ctrl+b then d
tmux attach -t usr2_infer
```

---

## 1) Set Up Environment and Dependencies

In [ ]:
# import os
# # Set this BEFORE importing torch or tensorflow
# os.environ["CUDA_VISIBLE_DEVICES"] = "1" # Choose which GPU and number of GPUs to make visible

# import torch 
# print(f"Current GPU: {torch.cuda.current_device()}") # Should return 0 (which is physically GPU 1)

In [ ]:
from pathlib import Path
import json
import os
from typing import Dict, Any
import datetime
import importlib
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from omegaconf import OmegaConf
import utils.inference_ as inference_module
from metrics import get_wer

inference_module = importlib.reload(inference_module)
transcribe = inference_module.transcribe

try:
    import wandb
    HAS_WANDB = True
except Exception:
    HAS_WANDB = False

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('wandb available:', HAS_WANDB)
print('inference.py reloaded for latest transcribe() changes')

torch: 2.5.1+cu121
cuda available: True
wandb available: True
inference.py reloaded for latest transcribe() changes


In [36]:
import wandb
wandb.login()

True

## 2) Define Configuration and Input

We define file paths, runtime options, and expected CSV schema.

In [37]:
from hydra import initialize_config_dir, compose

ROOT = Path(".").resolve()
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

with initialize_config_dir(config_dir=str((ROOT / "conf").resolve()), version_base=None):
    cfg = compose(config_name="config")

modality = "a"
detector = str(cfg.get("detector", "mediapipe"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_IN = ROOT / "csv/lrs2_test_modified.csv"
CSV_OUT = ROOT / f"csv/{modality}_test_with_predictions_{timestamp}.csv"
CHECKPOINT_PATH = ROOT / f"csv/{modality}_test_checkpoint_{timestamp}.json"

SAVE_EVERY = 20
MAX_ROWS = -1

assert "model" in cfg and "pretrained_model_path" in cfg.model, "Missing cfg.model.pretrained_model_path"
print("Model ckpt:", cfg.model.pretrained_model_path)

EXPECTED_COLUMNS = ["dataset", "video_path", "audio_path", "true_text", "status", "predicted_text", "wer"]

print("CSV_IN:", CSV_IN)
print("CSV_OUT:", CSV_OUT)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("modality:", modality, "detector:", detector, "device:", device)

Model ckpt: models/huge_high_resource_lrs2lrs3vox2avsp.pth
CSV_IN: /home/hainas-1/ong-ru/usr2.0/csv/lrs2_test_modified.csv
CSV_OUT: /home/hainas-1/ong-ru/usr2.0/csv/a_test_with_predictions_20260413_224431.csv
CHECKPOINT_PATH: /home/hainas-1/ong-ru/usr2.0/csv/a_test_checkpoint_20260413_224431.json
modality: a detector: mediapipe device: cuda


In [38]:
# tmux session setup for long-running inference
import subprocess

TMUX_SESSIONS = [f'usr2_infer_{modality}']

def ensure_tmux_session(name: str) -> None:
    exists = subprocess.run(
        ['tmux', 'has-session', '-t', name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    ).returncode == 0
    if not exists:
        subprocess.run(['tmux', 'new-session', '-d', '-s', name], check=True)

for session_name in TMUX_SESSIONS:
    ensure_tmux_session(session_name)

print('Ready tmux sessions:', ', '.join(TMUX_SESSIONS))
print(f'Attach with: tmux attach -t {TMUX_SESSIONS[0]}')
print('List with: tmux ls')

Ready tmux sessions: usr2_infer_a
Attach with: tmux attach -t usr2_infer_a
List with: tmux ls


## 3) Create Core Data Structures and Helper Utilities

In [39]:
def normalize_text(text: str) -> str:
    return ' '.join(str(text).strip().lower().split())

def load_checkpoint(path: Path) -> Dict[str, Any]:
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return {'processed': {}, 'last_idx': -1}

def save_checkpoint(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

def resolve_audio_path(audio_path_value: str) -> Path: # video_path->audio_path
    p = Path(str(audio_path_value).strip())
    if p.is_absolute():
        return p
    return (ROOT / p).resolve()

# quick sanity checks
assert normalize_text('  A   B  ') == 'a b'
print('Helper checks passed.')

Helper checks passed.


## 4) Implement the Main Processing Function

Load CSV, restore checkpoint/previous output, and run resumable batched inference.

In [42]:
def read_lrs2_test_csv(path: Path) -> pd.DataFrame:
    df_local = pd.read_csv(path, dtype="object", keep_default_na=False)
    missing = [col for col in ["dataset", "video_path", "audio_path", "true_text", 'status', 'prediction_text', 'wer'] if col not in df_local.columns]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}")
    return df_local

df = read_lrs2_test_csv(CSV_IN)
checkpoint_payload = load_checkpoint(CHECKPOINT_PATH)
processed_meta = dict(checkpoint_payload.get("processed", {}))
print("Loaded processed_meta:", len(processed_meta))

if MAX_ROWS > 0:
    df = df.head(MAX_ROWS).copy()

for c in ["prediction_words", "status"]:
    if c not in df.columns:
        df[c] = ""
    df[c] = df[c].astype("object")

if "wer" not in df.columns:
    df["wer"] = float("nan")
df["wer"] = pd.to_numeric(df["wer"], errors="coerce")

Loaded processed_meta: 0


## 5) Add Validation and Error Handling

In [43]:
assert CSV_IN.exists(), f'Missing input CSV: {CSV_IN}'
assert 'audio_path' in df.columns and 'true_text' in df.columns

missing_paths = 0
for ap in df['audio_path'].astype(str).head(50):
    if not resolve_audio_path(ap).exists():
        missing_paths += 1
print('Sample missing audio files (first 50 rows):', missing_paths)
print('Validation checks complete.')

Sample missing audio files (first 50 rows): 0
Validation checks complete.


## 6) Write Quick Unit Tests for Core Behavior

In [44]:
assert normalize_text('HELLO   WORLD') == 'hello world'
assert isinstance(load_checkpoint(Path('does_not_exist.json')), dict)
print('Quick tests passed.')

Quick tests passed.


## 7) Run an End-to-End Example in Notebook Cells

This cell supports resume/checkpointing + optional W&B logging.

In [45]:
from hydra import initialize_config_dir, compose

with initialize_config_dir(config_dir=str((ROOT / "conf").resolve()), version_base=None):
    cfg = compose(
        config_name="config",
        overrides=[
            "model/backbone=resnet_transformer_huge",
            "model.pretrained_model_path=models/huge_high_resource_lrs2lrs3vox2avsp.pth",
        ],
    )

print("backbone:", cfg.model.backbone)
print("pretrained_model_path:", cfg.model.pretrained_model_path)

backbone: {'_target_': 'espnet.nets.pytorch_backend.e2e_asr_transformer.E2E', 'odim': 1049, 'idim': 512, 'adim': 1280, 'aheads': 16, 'eunits': 5120, 'elayers': 32, 'ddim': '${model.backbone.adim}', 'dheads': '${model.backbone.aheads}', 'dunits': '${model.backbone.eunits}', 'dlayers': 9, 'gamma_init': 0.1}
pretrained_model_path: models/huge_high_resource_lrs2lrs3vox2avsp.pth


In [46]:
run = None
if HAS_WANDB:
    wandb.login(relogin=False)
    run = wandb.init(
        project="usr2-inference",
        name=f"{modality}_test_{timestamp}",
        settings=wandb.Settings(save_code=False),
        reinit=True,
    )

save_counter = 0

try:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Batched inference"):
        if str(df.at[idx, "status"]).strip().lower() == "ok":
            continue

        rel_audio = str(row["audio_path"]).strip()
        audio_path = resolve_audio_path(rel_audio)
        true_text = normalize_text(row["true_text"])

        if not audio_path.exists():
            msg = "audio file not found"
            df.at[idx, "prediction_words"] = msg
            df.at[idx, "wer"] = np.nan
            df.at[idx, "status"] = "error"
            processed_meta[rel_audio] = {"idx": int(idx), "status": "error", "error": msg}
            continue

        try:
            pred_text = transcribe(
                str(audio_path),
                cfg,
                modality=modality,
                device=device,
                detector=detector,
            )
            pred_norm = normalize_text(pred_text)
            row_wer = float(get_wer(pred_norm, true_text)) if true_text else np.nan

            df.at[idx, "prediction_words"] = pred_norm
            df.at[idx, "wer"] = row_wer
            df.at[idx, "status"] = "ok"

            processed_meta[rel_audio] = {"idx": int(idx), "status": "ok"}

            if run is not None and not np.isnan(row_wer):
                wandb.log({"wer_row": row_wer, "processed_rows": int((df["status"] == "ok").sum())})

        except Exception as exc:
            df.at[idx, "prediction_words"] = f"<error: {type(exc).__name__}: {exc}>"
            df.at[idx, "wer"] = np.nan
            df.at[idx, "status"] = "error"
            processed_meta[rel_audio] = {"idx": int(idx), "status": "error", "error": str(exc)}

        save_counter += 1
        if save_counter >= SAVE_EVERY:
            CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
            df.drop(columns=["error"], errors="ignore").to_csv(CSV_OUT, index=False)
            save_checkpoint(CHECKPOINT_PATH, {"processed": processed_meta, "last_idx": int(idx)})
            save_counter = 0

except KeyboardInterrupt:
    print("Interrupted by user. Saving resume state...")

finally:
    CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
    df.drop(columns=["error"], errors="ignore").to_csv(CSV_OUT, index=False)
    save_checkpoint(CHECKPOINT_PATH, {"processed": processed_meta})

    from metrics import WER
    wer_metric = WER()

    status_series = df["status"].astype(str).str.strip().str.lower()
    valid_mask = status_series != "error"
    eval_df = df.loc[valid_mask].copy()

    used_rows = 0
    for _, eval_row in eval_df.iterrows():
        pred_text = normalize_text(eval_row.get("prediction_words", ""))
        true_text = normalize_text(eval_row.get("true_text", ""))
        if not pred_text or not true_text:
            continue
        wer_metric.update(pred_text, true_text)
        used_rows += 1

    aggregate_wer = float(wer_metric.compute().item()) if used_rows > 0 else np.nan

    print("Saved output:", CSV_OUT)
    print("Saved checkpoint:", CHECKPOINT_PATH)
    print("Completed rows:", int((df["status"] == "ok").sum()), "/", len(df))
    print("Rows used for aggregate WER: ", used_rows)
    print("Aggregate WER:", aggregate_wer)

    if run is not None:
        wandb.log({"wer_aggregate": aggregate_wer, "completed_rows": int((df["status"] == "ok").sum())})
        run.finish()

df.head()

Batched inference: 100%|██████████| 1243/1243 [3:08:45<00:00,  9.11s/it] 

Saved output: /home/hainas-1/ong-ru/usr2.0/csv/a_test_with_predictions_20260413_224431.csv
Saved checkpoint: /home/hainas-1/ong-ru/usr2.0/csv/a_test_checkpoint_20260413_224431.json
Completed rows: 1243 / 1243
Rows used for aggregate WER:  1243
Aggregate WER: 0.01786786876618862


completed_rows,▁
processed_rows,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇███
wer_aggregate,▁
wer_row,▁▁▆▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▆▁▁▁▁▁▁
completed_rows,1243
processed_rows,1243
wer_aggregate,0.01787
wer_row,0


,dataset,video_path,audio_path,true_text,status,prediction_text,wer,prediction_words
0,lrs2,data/lrs2/main/6330311066473698535/00011.mp4,data/audio/6330311066473698535_00011.wav,AND FOR ME THE SURPRISE WAS,ok,,0.0,and for me the surprise was
1,lrs2,data/lrs2/main/6330311066473698535/00018.mp4,data/audio/6330311066473698535_00018.wav,THEY'RE MOVING AROUND,ok,,0.0,they're moving around
2,lrs2,data/lrs2/main/6330311066473698535/00022.mp4,data/audio/6330311066473698535_00022.wav,AND WE WERE RIGHT,ok,,0.0,and we were right
3,lrs2,data/lrs2/main/6330311066473698535/00025.mp4,data/audio/6330311066473698535_00025.wav,AND THE NEXT DAY,ok,,0.0,and the next day
4,lrs2,data/lrs2/main/6331559613336179781/00019.mp4,data/audio/6331559613336179781_00019.wav,WHEN THERE ISN'T MUCH ELSE IN THE GARDEN,ok,,0.0,when there isn't much else in the garden


In [ ]:
# quick sanity check on WER calculation
import jiwer

refs = ["hello world my name is rudy", "the quick brown fox", "lips are moving", "quick brown fox jumps over the stone"]
hyps = ["hello world my name rudy", "the quick brown", "lips are moving very fast", "quick"]

row_wer = [jiwer.wer(r, h) for r, h in zip(refs, hyps)]
number_of_rows = len(refs)
aggregate_wer = float(jiwer.wer(refs, hyps))
print('Row-wise WER:', row_wer)
print('Average WER:', sum(row_wer) / len(row_wer))
print('Aggregate WER:', aggregate_wer)

Row-wise WER: [0.16666666666666666, 0.25, 0.6666666666666666, 0.8571428571428571]
Average WER: 0.4851190476190476
Aggregate WER: 0.5


In [47]:
# Implementation of jiwer WER calculation for test set
def normalize_text_for_wer(text: str) -> str:
    return " ".join(str(text).strip().lower().split())

df = pd.read_csv(CSV_OUT, dtype="object", keep_default_na=False)

status_series = df["status"].astype(str).str.strip().str.lower()
valid_mask = status_series != "error"   # ignore error rows
df = df.loc[valid_mask]

refs = [normalize_text_for_wer(x) for x in df["true_text"].fillna("")]
hyps = [normalize_text_for_wer(x) for x in df["prediction_words"].fillna("")]

refs_concat = " ".join([text for text in refs if text])
hyps_concat = " ".join([text for text in hyps if text])

aggregate_wer = jiwer.wer(refs_concat, hyps_concat)

print(f"Aggregate WER : {aggregate_wer*100:.4f}% over {len(df)} valid rows")

Aggregate WER : 1.7868% over 1243 valid rows


In [49]:
# Implementation using own library metrics WER Calculations
from metrics import WER

wer_metric = WER()

status_series = df["status"].astype(str).str.strip().str.lower()
valid_mask = status_series != "error"   # ignore error rows
eval_df = df.loc[valid_mask].copy()

used_rows = 0
for _, row in eval_df.iterrows():
    pred_text = normalize_text(row.get("prediction_words", ""))
    true_text = normalize_text(row.get("true_text", ""))
    if not pred_text or not true_text:
        continue
    wer_metric.update(pred_text, true_text)
    used_rows += 1

aggregate_wer = 100 * float(wer_metric.compute().item()) if used_rows > 0 else float("nan")

print(f"Rows total: {len(df)}")
print(f"Rows ignored (status=error): {(~valid_mask).sum()}")
print(f"Rows used for corpus WER: {used_rows}")
print(f"Aggregate WER: {aggregate_wer:.4f}%")

Rows total: 1243
Rows ignored (status=error): 0
Rows used for corpus WER: 1243
Aggregate WER: 1.7868%


In [ ]:
# Inspect some error cases
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from hydra import initialize_config_dir, compose
from inference import transcribe
from metrics import get_wer

ROOT = Path("/home/hainas-1/ong-ru/usr2.0")
CSV_PATH = ROOT / f"csv/{modality}_test_with_predictions_{timestamp}.csv"


def normalize_text(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


df = pd.read_csv(CSV_PATH, dtype="object", keep_default_na=False)
df["wer"] = pd.to_numeric(df.get("wer", np.nan), errors="coerce")

with initialize_config_dir(config_dir=str((ROOT / "conf").resolve()), version_base=None):
    cfg = compose(
        config_name="config",
        overrides=[
            "model/backbone=resnet_transformer_huge",
            "model.pretrained_model_path=models/huge_high_resource_lrs2lrs3vox2avsp.pth",
        ],
    )

detector = str(cfg.get("detector", "mediapipe"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modality = "a"

for idx in ():
    if idx not in df.index:
        print(idx, "missing-index")
        continue
    if str(df.at[idx, "status"]).strip().lower() != "error":
        print(idx, f"skip-status={df.at[idx, 'status']}")
        continue

    rel_audio = str(df.at[idx, "audio_path"]).strip()
    audio_path = (ROOT / rel_audio).resolve() if not Path(rel_audio).is_absolute() else Path(rel_audio)
    true_text = normalize_text(df.at[idx, "true_text"])

    if not audio_path.exists():
        df.at[idx, "prediction_words"] = "audio file not found"
        df.at[idx, "wer"] = np.nan
        df.at[idx, "status"] = "error"
        print(idx, "error", "audio file not found")
        continue

    try:
        pred = transcribe(str(audio_path), cfg, modality=modality, device=device, detector=detector)
        pred_norm = normalize_text(pred)
        row_wer = float(get_wer(pred_norm, true_text)) if true_text else np.nan

        df.at[idx, "prediction_words"] = pred_norm
        df.at[idx, "wer"] = row_wer
        df.at[idx, "status"] = "ok"
        print(idx, "ok", row_wer)
    except Exception as exc:
        df.at[idx, "prediction_words"] = f"<error: {type(exc).__name__}: {exc}>"
        df.at[idx, "wer"] = np.nan
        df.at[idx, "status"] = "error"
        print(idx, "error", exc)

df.drop(columns=["error"], errors="ignore").to_csv(CSV_PATH, index=False)
print("saved:", CSV_PATH)

396 skip-status=ok
saved: /home/hainas-1/ong-ru/usr2.0/csv/a_test_with_predictions_20260413_174118.csv
